# Determinants of Life Expectancy: A Panel Data Analysis
This Notebook is a guided presentation of the reproducible pipeline. It must be run from the repository root. Essential logic remains in `src/`, and all interpretations are associational rather than causal.

## Research question and design
Which socioeconomic, healthcare, and public-health factors are associated with changes in national life expectancy, and how do those associations change after controlling for country and year fixed effects? The main comparison uses one complete-case sample across pooled OLS, year-effects, country-effects, and two-way fixed-effects models.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
root = Path.cwd()
assert (root / 'data/raw/life_expectancy.csv').exists(), 'Start Jupyter from the repository root'

## Data audit and cleaning
The raw file is never overwritten. Cleaning standardizes names in the processed data, preserves original values, creates documented analysis copies and flags, and does not use global mean imputation or automatic outlier deletion.

In [ ]:
subprocess.run([sys.executable, 'src/data_audit.py', '--input', 'data/raw/life_expectancy.csv'], check=True)
subprocess.run([sys.executable, 'src/data_cleaning.py'], check=True)
subprocess.run([sys.executable, 'src/descriptive_analysis.py'], check=True)

## Descriptive analysis
Review `reports/descriptive_results.md` and the corresponding tables and figures. Descriptive differences across countries or status groups are not interpreted causally.

## Model diagnostics and specification decisions
Adult mortality, infant deaths, under-five deaths, and contemporaneous HIV/AIDS mortality are excluded from the primary model because they are mortality outcomes or conceptually overlap life expectancy. Schooling and income composition are not combined in the primary model, and polio and diphtheria are compared through robustness checks. GDP retains its neutral label because its precise definition is unresolved.

In [ ]:
subprocess.run([sys.executable, 'src/model_diagnostics.py'], check=True)
subprocess.run([sys.executable, 'src/regression_models.py'], check=True)
subprocess.run([sys.executable, 'src/robustness.py'], check=True)

## Main results
Pooled coefficients combine cross-country and within-country associations. Country fixed-effects coefficients describe within-country associations after controlling for time-invariant country characteristics; two-way estimates additionally control for common year shocks. Country-clustered standard errors allow within-country residual dependence.

In [ ]:
main_results = pd.read_csv('tables/main_regression_results.csv')
sample_summary = pd.read_csv('tables/model_sample_summary.csv')
main_results.query("model == 'M4_two_way_fe'")[['variable', 'coefficient', 'std_error', 'ci_lower_95', 'ci_upper_95', 'p_value']]

## Robustness and interpretation boundary
The robustness checks address flagged values, transformations, uncertain definitions, alternative measures, influential observations, subgroup heterogeneity, temporal ordering, and covariance sensitivity. Complete-case selection is described separately. These checks do not establish causation. The complete interpretation is in `reports/final_report.md`; this Notebook intentionally does not duplicate that report.

In [ ]:
lagged_results = pd.read_csv('tables/lagged_model_results.csv')
selection_diagnostics = pd.read_csv('tables/complete_case_selection_diagnostics.csv')
lagged_results[['variable', 'coefficient', 'std_error', 'ci_lower_95', 'ci_upper_95', 'p_value']]